# Modelamiento Supervisado

**Caso de Estudio:** Analítica de Clientes — Predicción de Abandono  
**Asignatura:** SCY1101 — Programación para la Ciencia de Datos  
**Evaluación Parcial N°2**

---

## Descripción

Este notebook implementa el ciclo completo de **modelamiento supervisado**, abordando dos sub-problemas a partir del mismo dataset:

1. **Regresión:** Predicción de `score_crediticio` mediante `LinearRegression` y `DecisionTreeRegressor`.
2. **Clasificación:** Predicción de `abandono` mediante `LogisticRegression`, `DecisionTreeClassifier` y `SVC` (SVM).

Cada modelo se construye dentro de un **`Pipeline` de Scikit-learn**, garantizando reproducibilidad, correcta separación entre preprocesamiento de entrenamiento y prueba, y ausencia de *data leakage*. Para los modelos de clasificación se establece una línea base mediante **validación cruzada estratificada** (`StratifiedKFold`), reportando múltiples métricas.

---

## Requisitos de Software

- `pandas >= 1.1.0`, `numpy >= 2.0.0`
- `scikit-learn >= 1.3` — Pipeline, ColumnTransformer, modelos, validación cruzada

In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [ ]:
data = pd.read_csv('../data/dataset_clientes.csv')
data.head()

,id_cliente,fecha_registro,edad,genero,region,estado_civil,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,...,ultima_compra_dias,uso_app,tipo_plan,num_productos,tiene_tarjeta_credito,canal_registro,dia_semana_registro,hora_registro,codigo_postal,abandono
0,1,2021-10-27,66,Otro,Norte,Divorciado,9.243057e+05,524088.303055,2.448145e+06,455.406680,...,356,Bajo,Estandar,3,1,Tienda,Lunes,22,3824,1
1,2,2018-08-25,51,Masculino,Centro,Soltero,1.384687e+06,314259.751474,1.620569e+06,575.048508,...,307,Medio,Premium,4,1,App,Martes,10,4148,0
2,3,2019-05-25,48,Femenino,Norte,Casado,NaN,387192.316142,5.395040e+06,770.716904,...,232,Alto,Premium,4,1,App,Jueves,6,7200,0
3,4,2022-04-20,54,Masculino,Sur,Casado,4.369032e+05,417328.601856,2.999350e+06,442.722671,...,165,Alto,Estandar,2,1,App,Domingo,16,1782,1
4,5,2020-03-19,31,Otro,Centro,Soltero,7.408561e+05,490961.191253,1.637711e+06,468.188403,...,283,Bajo,Estandar,3,1,Web,Martes,8,3448,1


In [ ]:
# Se eliminan los duplicados
data = data.drop_duplicates()

# 1. Preparación de Datos y Diseño del Pipeline

## 1.1 Arquitectura del Pipeline

Se utiliza `sklearn.pipeline.Pipeline` combinado con `ColumnTransformer` para separar el preprocesamiento de variables numéricas y categóricas. Esta arquitectura garantiza:

- **Sin data leakage:** Los parámetros de preprocesamiento (media para imputación, percentiles para winsorización, estadísticas de escalado) se aprenden **únicamente** sobre el conjunto de entrenamiento y se aplican al conjunto de prueba.
- **Reproducibilidad:** El pipeline encapsula todo el flujo, permitiendo serializar y desplegar el modelo completo con `joblib.dump()`.
- **Coherencia en producción:** En producción, los datos nuevos pasarán exactamente por la misma transformación que los datos de entrenamiento.

## 1.2 Transformadores Personalizados

Se implementan tres transformadores que extienden la interfaz `BaseEstimator + TransformerMixin`:

| Transformador | Propósito | Justificación |
|---|---|---|
| `Winsorizer` | Recorte de outliers por percentil | Evita que valores extremos sesguen coeficientes lineales sin eliminar filas |
| `CorrelationFilter` | Elimina variables con r > 0.9 | Reduce multicolinealidad, que infla varianza de coeficientes en LinearRegression |
| `DataFrameConverter` | Convierte array a DataFrame con nombres de columna | Necesario para que `CorrelationFilter` opere con nombres de columna |

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Tratamiento de atípicos via recorte por percentiles.
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns_ = X.columns
        else:
            self.columns_ = np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        for col in self.columns_:
            lower = X[col].quantile(self.limits[0])
            upper = X[col].quantile(1 - self.limits[1])
            X = X.astype('float64')
            X[col] = np.clip(X[col], lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        return np.array(input_features)

In [ ]:
def tratar_duplicados(X: pd.DataFrame, drop: bool = True) -> pd.DataFrame:
    """
    Tratamiento de duplicados.
    Si drop=True elimina filas duplicadas, si no las deja.
    """
    return X.drop_duplicates() if drop else X

In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad).
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.drop(columns=self.columns_to_drop_, errors='ignore').values

In [ ]:
class DataFrameConverter(BaseEstimator, TransformerMixin):
    """
    Convierte el array de ColumnTransformer en DataFrame con nombres de columnas.
    """
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_names_ = None

    def fit(self, X, y=None):
        self.feature_names_ = self.preprocessor.get_feature_names_out()
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

# 2. Modelos de Regresión

## 2.1 Justificación del Sub-problema de Regresión

El EDA reveló que `score_crediticio` es una variable continua con distribución aproximadamente normal (μ ≈ 600, σ ≈ 100). Predecir el score crediticio de un cliente es relevante para la segmentación de riesgo y el diseño de ofertas personalizadas.

Se implementan dos modelos obligatorios:

| Modelo | Supuestos | Trade-offs |
|---|---|---|
| **LinearRegression** | Relación lineal entre features y target; homoscedasticidad; sin multicolinealidad | Alta interpretabilidad; sensible a outliers y multicolinealidad; no captura relaciones no lineales |
| **DecisionTreeRegressor** | Sin supuestos distribucionales | Captura no linealidades; propenso a sobreajuste si no se regulariza (poda mediante `max_depth`) |

**Métricas de evaluación para regresión:**
- **R² (Coeficiente de determinación):** Proporción de varianza explicada por el modelo. R²=1 es perfecto; R²=0 equivale a predecir siempre la media.
- **MAE (Error Absoluto Medio):** Interpretable en las mismas unidades que el target; robusta ante outliers.
- **RMSE (Raíz del Error Cuadrático Medio):** Penaliza más los errores grandes; sensible a outliers.

> Se utilizará score_crediticio como target

In [ ]:
target_reg = 'score_crediticio'

features_num = [
    'edad', 'ingreso_mensual', 'gasto_mensual', 'deuda_total',
    'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos'
]
features_cat = [
    'genero', 'region', 'estado_civil', 'uso_app', 'tipo_plan', 'canal_registro'
]

X_reg = data[features_num + features_cat]
y_reg = data[target_reg]

mask = y_reg.notna()
X_reg, y_reg = X_reg[mask], y_reg[mask]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=44
)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

### 2.2 LinearRegression

La Regresión Lineal busca los coeficientes β que minimizan la Suma de Cuadrados de los Residuales (OLS). Es el modelo de referencia (*baseline*) para regresión por su alta interpretabilidad.

**Decisiones de configuración:**
- Se incluye un `CorrelationFilter` para eliminar features con correlación > 0.9, mitigando la **multicolinealidad** que inflaría los errores estándar de los coeficientes.
- `StandardScaler` se aplica a las variables numéricas para mejorar la convergencia numérica en OLS con muchas features.
- No se fija `fit_intercept=False` ya que no existe razón de negocio para asumir que el score es 0 cuando todos los predictores son 0.

In [ ]:
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

modelo_lr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_lr),
    ('conversion',    DataFrameConverter(preprocessor_lr)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LinearRegression())
])

modelo_lr.fit(X_train_reg, y_train_reg)
y_pred_lr = modelo_lr.predict(X_test_reg)

r2_lr   = r2_score(y_test_reg, y_pred_lr)
mae_lr  = mean_absolute_error(y_test_reg, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))

print('LinearRegression')
print(f"  R2  : {r2_lr:.4f}")
print(f"  MAE : {mae_lr:,.2f}")
print(f"  RMSE: {rmse_lr:,.2f}")

LinearRegression
  R2  : 0.0009
  MAE : 79.45
  RMSE: 98.92


### 2.3 DecisionTreeRegressor

El Árbol de Decisión para regresión divide recursivamente el espacio de features minimizando el MSE en cada nodo. No requiere supuestos distribucionales y captura relaciones no lineales naturalmente.

**Decisiones de configuración (hiperparámetros base):**
- `max_depth=2`: Limita la profundidad del árbol para evitar sobreajuste severo en la fase baseline. Se explorará mayor profundidad en la optimización de hiperparámetros.
- `min_samples_leaf=200`: Un nodo hoja debe tener al menos 200 muestras, forzando predicciones más estables y generalizables.
- `min_samples_split=100`: Un nodo interno solo se divide si tiene al menos 100 muestras.
- `random_state=44`: Semilla fija para reproducibilidad.

**Nota:** Los R² cercanos a 0 en ambos modelos indican que `score_crediticio` no puede predecirse linealmente con las features disponibles, lo cual es coherente con los bajos valores de correlación detectados en el EDA. Esta observación tiene valor diagnóstico y será analizada en profundidad en el notebook de evaluación.

In [ ]:

modelo_dtr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion',    DataFrameConverter(preprocessor)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeRegressor(
                          max_depth=2, min_samples_leaf=200,
                          min_samples_split=100, random_state=44))
])

modelo_dtr.fit(X_train_reg, y_train_reg)
y_pred_dtr = modelo_dtr.predict(X_test_reg)

r2_dtr   = r2_score(y_test_reg, y_pred_dtr)
mae_dtr  = mean_absolute_error(y_test_reg, y_pred_dtr)
rmse_dtr = np.sqrt(mean_squared_error(y_test_reg, y_pred_dtr))

print('DecisionTreeRegressor')
print(f"  R2  : {r2_dtr:.4f}")
print(f"  MAE : {mae_dtr:,.2f}")
print(f"  RMSE: {rmse_dtr:,.2f}")

DecisionTreeRegressor
  R2  : 0.0008
  MAE : 79.45
  RMSE: 98.92


# 3. Modelos de Clasificación

## 3.1 Justificación del Sub-problema de Clasificación

La variable `abandono` es binaria (0/1), lo que define un problema de **clasificación binaria supervisada**. El objetivo de negocio es **detectar a los clientes que van a abandonar** para activar acciones preventivas de retención.

**Elección de métrica principal — Recall:**
> En el contexto del abandono de clientes, el **costo de un Falso Negativo** (no detectar un cliente que va a abandonar) es mayor que el de un Falso Positivo (ofrecer retención a un cliente que se quedaría de igual forma). Por tanto, se prioriza el **Recall** como métrica de optimización, complementado con F1-score para balancear precisión y exhaustividad.

Se implementan los tres modelos requeridos por la rúbrica:

| Modelo | Tipo | Justificación de selección |
|---|---|---|
| **LogisticRegression** | Lineal | Modelo paramétrico interpretable; coeficientes expresan el log-odds de abandono por unidad de cambio en cada feature; apropiado como baseline de clasificación |
| **DecisionTreeClassifier** | No lineal | Captura interacciones entre variables; altamente interpretable (visualizable como árbol); puede sobreajustar sin regularización |
| **SVC (SVM)** | No lineal (kernel RBF) | Maximiza el margen de separación entre clases; efectivo en espacios de alta dimensión; el kernel RBF captura fronteras de decisión complejas |

## 3.2 Estrategia de Validación

Se utiliza **validación cruzada estratificada** (`StratifiedKFold(n_splits=5)`) en lugar de una simple partición train/test. Esto garantiza que cada fold mantenga la misma proporción de clases que el dataset completo, produciendo estimaciones de desempeño más robustas y menos sesgadas por la distribución aleatoria de las muestras.

## 3.3 Tratamiento del Desbalance de Clases

El EDA reveló desbalance en `abandono`. Se aplica `class_weight='balanced'` en `LogisticRegression` y `SVC`, lo que ajusta automáticamente los pesos de las muestras inversamente proporcional a la frecuencia de cada clase, obligando al modelo a poner más atención en los clientes que abandonan.

> Se utilizará abandono como target

Se priorizará el estadístico recall esto debido a que queremos priorizar los falsos positivos.

In [ ]:
target_cls = 'abandono'

X_cls = data[features_num + features_cat]
y_cls = data[target_cls]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=29, stratify=y_cls
)

### 3.4 DecisionTreeClassifier (Línea Base)

El Árbol de Decisión para clasificación construye reglas `if-then-else` maximizando la **pureza de los nodos** (medida mediante Gini o entropía). En la fase baseline se entrena sin restricciones de profundidad para observar el comportamiento del modelo en su estado "natural" (propenso a sobreajuste).

**Configuración baseline:**
- `random_state=29`: Reproducibilidad del algoritmo de split.
- Sin `class_weight` en baseline (se ajustará en la optimización).
- Sin `max_depth` explícito (árbol completo — referencia de sobreajuste).

In [ ]:
pipeline_dtc = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeClassifier(random_state=29))
])

cv_dtc = cross_validate(
    pipeline_dtc, X_train_cls, y_train_cls,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print('DecisionTreeClassifier')
print(f"  Accuracy : {cv_dtc['test_accuracy'].mean():.4f}")
print(f"  F1       : {cv_dtc['test_f1'].mean():.4f}")
print(f"  Precision: {cv_dtc['test_precision'].mean():.4f}")
print(f"  Recall   : {cv_dtc['test_recall'].mean():.4f}")

DecisionTreeClassifier
  Accuracy : 0.5619
  F1       : 0.4539
  Precision: 0.4491
  Recall   : 0.4589


### 3.5 LogisticRegression

La Regresión Logística modela la probabilidad de abandono como una función sigmoide aplicada a una combinación lineal de los features. Es el modelo de referencia para clasificación binaria por su interpretabilidad: cada coeficiente representa el cambio en el **log-odds** de abandono por unidad de cambio en la variable.

**Configuración:**
- `class_weight='balanced'`: Compensa el desbalance de clases (justificado en §3.3).
- `max_iter=10000`: El optimizador (LBFGS por defecto) requiere más iteraciones con muchas features. Se aumenta para garantizar convergencia.
- `random_state=29`: Reproducibilidad.
- **Requiere escalado:** Las variables numéricas se escalan con `StandardScaler` dentro del pipeline, ya que la Regresión Logística es sensible a diferencias de escala entre features.

In [ ]:
pipeline_logreg = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LogisticRegression(class_weight='balanced', max_iter=10000, random_state=29))
])

cv_logreg = cross_validate(
    pipeline_logreg, X_train_cls, y_train_cls,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print('LogisticRegression')
print(f"  Accuracy : {cv_logreg['test_accuracy'].mean():.4f}")
print(f"  F1       : {cv_logreg['test_f1'].mean():.4f}")
print(f"  Precision: {cv_logreg['test_precision'].mean():.4f}")
print(f"  Recall   : {cv_logreg['test_recall'].mean():.4f}")

LogisticRegression
  Accuracy : 0.6255
  F1       : 0.5689
  Precision: 0.5238
  Recall   : 0.6229


### 3.6 Support Vector Machine (SVM — SVC con kernel RBF)

Las Máquinas de Vectores de Soporte buscan el **hiperplano de máximo margen** que separa las clases. Con el kernel RBF (*Radial Basis Function*), el modelo proyecta los datos a un espacio de mayor dimensionalidad donde puede encontrar fronteras de decisión no lineales.

**Justificación del kernel RBF:**
El EDA mostró que las relaciones entre features y abandono son complejas y no lineales (correlaciones bajas en general). El kernel RBF es el kernel más versátil y apropiado como punto de partida cuando no se conoce la estructura del espacio de features.

**Configuración:**
- `kernel='rbf'`: Kernel radial para fronteras de decisión no lineales.
- `probability=True`: Habilita la estimación de probabilidades (necesario para calcular ROC-AUC).
- `class_weight='balanced'`: Tratamiento del desbalance de clases.
- `random_state=29`: Reproducibilidad.
- **Requiere escalado obligatorio:** SVM es extremadamente sensible a la escala de features. Se utiliza `StandardScaler` en el preprocesador dedicado `preprocessor_svm`.

In [ ]:
numeric_transformer_svm = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

preprocessor_svm = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_svm, features_num),
        ('cat', categorical_transformer,  features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

pipeline_svm = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_svm),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=29))
])

cv_svm = cross_validate(
    pipeline_svm, X_train_cls, y_train_cls,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print('SVM')
print(f"  Accuracy : {cv_svm['test_accuracy'].mean():.4f}")
print(f"  F1       : {cv_svm['test_f1'].mean():.4f}")
print(f"  Precision: {cv_svm['test_precision'].mean():.4f}")
print(f"  Recall   : {cv_svm['test_recall'].mean():.4f}")

SVM
  Accuracy : 0.6137
  F1       : 0.5638
  Precision: 0.5108
  Recall   : 0.6296


## 4. Resumen de Resultados de la Línea Base (Clasificación — Validación Cruzada)

| Modelo | Accuracy | F1 | Precision | Recall |
|---|---|---|---|---|
| DecisionTreeClassifier | 0.5619 | 0.4539 | 0.4491 | 0.4589 |
| LogisticRegression | 0.6255 | 0.5689 | 0.5238 | 0.6229 |
| SVM (RBF) | 0.6137 | 0.5638 | 0.5108 | 0.6296 |

**Interpretación:**
- **LogisticRegression y SVM** dominan en Recall (~0.62-0.63), lo cual es la métrica prioritaria para el negocio. El árbol de decisión sin poda presenta el rendimiento más bajo, evidenciando sobreajuste al entrenamiento.
- La diferencia entre Accuracy y Recall refleja el efecto del `class_weight='balanced'`: los modelos sacrifican algo de accuracy global para detectar mejor la clase minoritaria (clientes que abandonan).
- Estos resultados constituyen la **línea base** que se buscará superar mediante la optimización de hiperparámetros en el notebook `04_hyperparameter_optimization.ipynb`.